---

# Fishora: CV Evaluation Suite

*Every exported Fishora classifier through the same robustness, field and out-of-distribution slices.*

---

## Fishora Team

- Member 1
- Member 2
- Member 3

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Workspace**](#3)
4. [**Suite Runs**](#4)
5. [**CPU Latency**](#5)
6. [**Reports**](#6)
7. [**Export**](#7)


---

# Introduction <a name="1"></a>

---

## Overview

*The Fishora classifier scores 100% on its held-out test split, but that split shares photographing conditions with training: the three Roboflow classes (gembolo, tenggiri, tuna) are always on red cloth and the eight Fish-gres classes mostly on a gray tray. A perfect in-distribution score therefore says little about a landing site.*

## Aim

*Run the repository's evaluation suite (`evaluation/cv/cv_suite.py`) on a GPU for every exported model and produce the per-run reports and the cross-model comparison used in the Evaluation Artifact. The suite code is the same file that runs locally; this notebook only assembles the workspace and calls it.*

## Metric

***Accuracy and macro F1-score** per slice, with bootstrap 95% confidence intervals on the headline slices. Macro F1 is used because the classes are imbalanced.*

***Wrong-and-confident rate** is the share of predictions that are wrong with confidence of at least 0.9. It measures the failures the human-verification gate cannot catch.*

***OOD AUROC and FPR at 95% TPR** measure whether the max-softmax confidence separates in-scope fish from out-of-distribution images.*

$$\text{FPR@95} = P\left(\text{conf}_{\text{OOD}} \geq t_{95}\right), \quad P\left(\text{conf}_{\text{ID}} \geq t_{95}\right) = 0.95$$

## Dataset

*Two inputs are combined. The Fishora dataset on Kaggle supplies the 849 held-out test images. The private `fishora_eval_kit` dataset, packed by `evaluation/cv/kaggle/pack_kit.py`, supplies the suite code, the locked screening file, 258 field photos, 172 out-of-distribution photos (iNaturalist and Openverse, Creative Commons, attribution in `attribution.csv`), and the fish masks generated locally with rembg.*

*The datasets are used only for testing. No model is trained or tuned in this notebook.*

```
fishora_eval_kit/
├── evaluation/cv/   suite code, protocol/ (locked screening and folds)
├── apps/cv_service/ export loader shared with the CV service
└── evaluation/cv/data/
     ├── field/      in-class photos from outside the training sources
     ├── ood/        unknown fish species and non-fish objects
     ├── masks/      one fish mask per test image
     └── dataset/metadata/manifest.csv
```

## Approach: Slice-Based Evaluation

*Each model is run once over every slice and all eleven calibrated probabilities are stored. Metrics are computed afterwards from those stored outputs, so a new metric never requires another pass over the images.*

```
test image (849) --+-- clean
                   +-- 7 corruptions x 3 severities
                   +-- bg_removed / fish_erased / bg_swap   (masks)
field photos ------+-- field
OOD photos --------+-- ood_unknown_fish / ood_nonfish / ood_synthetic
                          |
                   predictions.csv -> report.py -> metrics + figures
```

*Live fish under water are out of the product's operating domain and are not scored; the exclusion is recorded in every `run.json`.*

---

# Initialization <a name="2"></a>

---

## Environment Setup

Enable a GPU (T4 or P100) in the session settings and add three inputs: the Fishora dataset, the private `fishora_eval_kit` dataset, and the output of the linear-probe notebook. Runs whose `predictions.csv` already exists are skipped, so a re-run after a disconnect resumes. Expected runtime on a T4 is about 30 minutes for six models plus 10 minutes of CPU latency.

In [ ]:
!nvidia-smi

The following cell installs all libraries used in this notebook.

In [ ]:
%pip install -q -U "timm>=1.0.20"

## Import Libraries

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import Image as ShowImage, Markdown, display

print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Seed Everything

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(42)

## Settings

Paths and run options are centralised here. `RUN_ORDER` fixes the order of the comparison table; models that are not found are skipped with a message. The retrained ViT-L (`lp_vit_l`) stands in for the production export as the baseline: it reproduces the production recipe (temperature 0.0771 against 0.0773, 100% test for both), and it keeps the comparison inside one training run. `LIMIT` above zero restricts the test split to its first N images and is meant only for a quick dry run.

In [ ]:
class Settings:
    SEED      = 42
    INPUT     = Path('/kaggle/input')
    WORKSPACE = Path('/kaggle/working/fishora')
    BATCH     = 64
    LIMIT     = 0
    RUN_LATENCY = True
    RUN_ORDER = [
        'lp_vit_l', 'lp_vit_b', 'lp_vit_s',
        'lp_convnext_t', 'lp_efficientnet_b0', 'lp_mobilenetv3_l',
    ]
    XF_SKIP_CORRUPTIONS = True
    LP_NAMES = {
        'vit_l_dinov3': 'lp_vit_l', 'vit_b_dinov3': 'lp_vit_b', 'vit_s_dinov3': 'lp_vit_s',
        'convnext_t_dinov3': 'lp_convnext_t', 'efficientnet_b0': 'lp_efficientnet_b0',
        'mobilenetv3_l': 'lp_mobilenetv3_l',
    }

CFG = Settings()
print(sorted(p.name for p in CFG.INPUT.iterdir()))

## Load Dataset

Three things are located under `/kaggle/input`: the kit (a directory holding `evaluation/cv/protocol/field_folds.csv` and the masks, so an older workspace copied in with reused results is never picked), the Fishora dataset (a directory holding `cleaned/` and `splits/`), and every model export (a directory holding `inference_config.json`).

In [ ]:
def find_dirs(predicate):
    return sorted({p for p in CFG.INPUT.rglob('*') if p.is_dir() and predicate(p)}, key=lambda p: len(str(p)))

KIT = find_dirs(lambda p: (p / 'evaluation' / 'cv' / 'protocol' / 'field_folds.csv').exists() and (p / 'evaluation' / 'cv' / 'data' / 'masks').is_dir())[0]
DATASET = find_dirs(lambda p: (p / 'cleaned').is_dir() and (p / 'splits').is_dir())[0]

exports = {}
for cfg_path in CFG.INPUT.rglob('inference_config.json'):
    cfg = json.loads(cfg_path.read_text())
    folder = cfg_path.parent.parent.name
    fold = cfg.get('heldout_fold')
    if cfg.get('gate'):
        run = f'gated_vit_b_{fold}' if fold else 'gated_vit_b_final'
    elif cfg.get('field_training') == 'all':
        run = 'final_vit_b'
    elif fold:
        base = CFG.LP_NAMES.get(folder.removesuffix(f'_xf{fold}'))
        run = base and base.replace('lp_', 'xf_', 1) + f'_{fold}'
    elif 'recipe' in cfg:
        run = CFG.LP_NAMES.get(folder)
    else:
        run = 'baseline_vit_l_prod'
    if run and run not in exports:
        exports[run] = cfg_path.parent

XF_BASES = sorted({r.rsplit('_', 1)[0] for r in exports if r.startswith('xf_')},
                  key=lambda r: CFG.RUN_ORDER.index(r.replace('xf_', 'lp_', 1)))
FOLD_BASES = XF_BASES + (['gated_vit_b'] if any(r.startswith('gated_vit_b_') and r[-2:] in ('_A', '_B') for r in exports) else [])
FINALS = [r for r in ('final_vit_b', 'gated_vit_b_final') if r in exports]
RUNS = CFG.RUN_ORDER + [f'{b}_{f}' for b in FOLD_BASES for f in ('A', 'B')] + FINALS

print('kit     :', KIT)
print('dataset :', DATASET)
for run in RUNS:
    print(f'{run:22s}', exports.get(run, 'NOT FOUND'))

---

# Workspace <a name="3"></a>

---

The kit is copied into a writable workspace that mirrors the repository, and the Fishora test images are linked in at the path the suite expects (`evaluation/cv/data/dataset/cleaned`). The suite refuses to run if `evaluation/cv/protocol/screening.csv` no longer matches the hash in `evaluation/cv/protocol/screening.lock`, which is checked here once before any model runs.

In [ ]:
if not (CFG.WORKSPACE / 'evaluation' / 'cv' / 'cv_suite.py').exists():
    shutil.copytree(KIT, CFG.WORKSPACE, dirs_exist_ok=True)
link = CFG.WORKSPACE / 'evaluation' / 'cv' / 'data' / 'dataset' / 'cleaned'
if not link.exists():
    link.symlink_to(DATASET / 'cleaned', target_is_directory=True)

results_dir = CFG.WORKSPACE / 'evaluation' / 'cv' / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
old_layout = list(CFG.INPUT.rglob('eval/results/*/predictions.csv'))
for pred in [*CFG.INPUT.rglob('evaluation/cv/results/*/predictions.csv'), *old_layout]:
    target = results_dir / pred.parent.name
    if not target.exists():
        shutil.copytree(pred.parent, target)
        print('reused previous results:', pred.parent.name)
for lat in [*CFG.INPUT.rglob('evaluation/cv/results/latency.json'), *CFG.INPUT.rglob('eval/results/latency.json')]:
    if not (results_dir / 'latency.json').exists():
        shutil.copy(lat, results_dir / 'latency.json')

sys.path.insert(0, str(CFG.WORKSPACE))
os.chdir(CFG.WORKSPACE)
from evaluation.cv.cv_suite import verify_screening_lock
from evaluation.cv.common import load_split

print('screening sha256:', verify_screening_lock())
test = load_split('test')
missing = [s.path for s in test if not s.path.exists()]
print(f'test images: {len(test)}  missing: {len(missing)}')
print('masks:', len(list((CFG.WORKSPACE / 'evaluation/cv/data/masks').glob('*.png'))))

---

# Suite Runs <a name="4"></a>

---

Each model runs in a fresh Python process through the same command a local run uses, so the results are byte-for-byte comparable with `bash evaluation/cv/run_all.sh`. Progress lines are streamed; the final line of each run lists item counts per slice.

In [ ]:
def run(cmd):
    env = {**os.environ, 'FISHORA_CV_DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu'}
    proc = subprocess.Popen(cmd, cwd=CFG.WORKSPACE, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'failed: {cmd}')

done = []
for name in RUNS:
    already = (CFG.WORKSPACE / 'evaluation/cv/results' / name / 'predictions.csv').exists()
    if name not in exports and not already:
        print(f'== {name}: export not found, skipped')
        continue
    if (CFG.WORKSPACE / 'evaluation/cv/results' / name / 'predictions.csv').exists():
        print(f'== {name}: already done')
    else:
        print(f'== {name}')
        fold_run = name.startswith(('xf_', 'gated_vit_b_')) and name[-2:] in ('_A', '_B')
        extra = ['--skip-corruptions'] if fold_run and CFG.XF_SKIP_CORRUPTIONS else []
        run([sys.executable, '-u', '-m', 'evaluation.cv.cv_suite', '--run', name, '--export', str(exports[name]),
             '--batch', str(CFG.BATCH), '--limit', str(CFG.LIMIT), *extra])
    run([sys.executable, '-m', 'evaluation.cv.report', name])
    done.append(name)

---

# CPU Latency <a name="5"></a>

---

Latency is measured on the session CPU with four threads, batch 1, 10 warm-up and 100 timed calls of the production `predict()`, one subprocess per model. Models already in a reused `latency.json` are not measured again, and cross-fitted models reuse the latency of the same architecture, since only the head weights differ. A gated export is timed through its own `predict()`, so its latency includes both gates. The absolute numbers belong to this Kaggle CPU, not to an operator's device, but every model is measured on the same machine, so the ratios between models are what the comparison uses.

In [ ]:
lat_path = CFG.WORKSPACE / 'evaluation/cv/results/latency.json'
measured = json.loads(lat_path.read_text())['models'] if lat_path.exists() else {}
todo = [n for n in done if n in exports and n[-2:] not in ('_A', '_B') and n not in measured]
if CFG.RUN_LATENCY and todo:
    run([sys.executable, '-u', '-m', 'evaluation.cv.bench_latency', *[f'{n}={exports[n]}' for n in todo]])
    new = json.loads(lat_path.read_text())
    new['models'] = {**measured, **new['models']}
    lat_path.write_text(json.dumps(new, indent=2))
if lat_path.exists():
    display(pd.DataFrame(json.loads(lat_path.read_text())['models']).T)

---

# Reports <a name="6"></a>

---

The baseline report comes first because it defines the failures the Evaluation Artifact starts from. The comparison then places every model on the slices that separate them.

## Baseline Summary

The summary lists every labelled slice with its confidence interval, the background tests, the out-of-distribution table, and field accuracy per class.

In [ ]:
base = CFG.WORKSPACE / 'evaluation/cv/results/lp_vit_l'
display(Markdown((base / 'summary.md').read_text()))
for fig in ['confusion', 'corruptions', 'confidence', 'reliability']:
    display(ShowImage(str(base / 'figures' / f'{fig}.png')))

#### Insights

> Fill in after the run: the gap between clean and field accuracy, the shortcut rate on fish_erased against the 9.1% chance level, and the out-of-distribution acceptance at the production threshold of 0.0.

> Each finding should name the condition, the evidence, and the metric that hides it.

## Model Comparison

The comparison table and the Pareto figure put quality on the separating slices against CPU latency. The in-distribution panel is included to show that it cannot rank the models.

In [ ]:
merged = [b for b in FOLD_BASES if all(f'{b}_{f}' in done for f in ('A', 'B'))]
if merged:
    run([sys.executable, '-m', 'evaluation.cv.merge_xfit', *merged])
    for b in merged:
        run([sys.executable, '-m', 'evaluation.cv.report', b])
lp_runs = [n for n in CFG.RUN_ORDER if n in done]
run([sys.executable, '-m', 'evaluation.cv.report', '--compare', *lp_runs, *merged, *[f for f in FINALS if f in done]])
for p in sorted((CFG.WORKSPACE / 'evaluation/cv/results/compare').glob('*_pareto.png')):
    display(ShowImage(str(p)))

#### Insights

> Fill in after the run: which backbones sit on the Pareto front for field macro-F1 and background-swap accuracy, and how much quality each step down in latency costs.

---

# Export <a name="7"></a>

---

Only the results folder is archived: predictions, metrics, summaries, figures, and latency. It contains no photographs, so it can be committed to the repository under `evaluation/cv/results/`.

In [ ]:
archive = shutil.make_archive('/kaggle/working/fishora_eval_results', 'zip',
                              root_dir=CFG.WORKSPACE / 'evaluation' / 'cv', base_dir='results')
print(archive, f'{Path(archive).stat().st_size / 2**20:.1f} MB')